# AI Investment Dashboard — Model Exploration

Notebook for prototyping and validating key models before integrating into the Streamlit app.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import sys; sys.path.insert(0, '..')

from src.data_loader import fetch_price_data
from src.optimizer import max_sharpe_weights, efficient_frontier
from src.models import black_scholes, monte_carlo_paths, var_cvar, gmm_scenario_returns

plt.style.use('dark_background')

## 1. Data Loading

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
START, END = '2021-01-01', '2024-12-31'

prices = fetch_price_data(tuple(TICKERS), START, END)
returns = prices.pct_change().dropna()
print(f'Shape: {prices.shape}')
prices.tail()

## 2. Portfolio Optimization — Efficient Frontier

In [ ]:
frontier = efficient_frontier(returns, n_portfolios=2000, risk_free_rate=0.045)
weights, ret, vol, sharpe = max_sharpe_weights(returns, risk_free_rate=0.045)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(frontier['vols']*100, frontier['rets']*100,
                c=frontier['sharpes'], cmap='viridis', alpha=0.5, s=10)
ax.scatter(vol*100, ret*100, color='red', s=200, zorder=5, label='Max Sharpe')
plt.colorbar(sc, label='Sharpe Ratio')
ax.set_xlabel('Volatility (%)'); ax.set_ylabel('Return (%)')
ax.set_title('Efficient Frontier'); ax.legend()
plt.tight_layout(); plt.show()

print('\nOptimal Weights:')
for t, w in zip(TICKERS, weights):
    print(f'  {t}: {w*100:.1f}%')
print(f'Sharpe: {sharpe:.3f} | Return: {ret*100:.1f}% | Vol: {vol*100:.1f}%')

## 3. Black-Scholes Option Pricing

In [ ]:
call, put, greeks = black_scholes(S=150, K=155, T=0.5, r=0.045, sigma=0.25)
print(f'Call: ${call:.4f} | Put: ${put:.4f}')
print('Greeks:', {k: round(v, 4) for k, v in greeks.items() if k not in ['d1','d2']})

## 4. VaR, CVaR & Monte Carlo

In [ ]:
port_ret = returns @ weights
var95, cvar95 = var_cvar(port_ret, 0.95)
print(f'VaR 95%: {var95*100:.2f}% | CVaR 95%: {cvar95*100:.2f}%')

paths = monte_carlo_paths(port_ret, n_paths=1000, horizon=252)
final = paths[-1]
print(f'Simulated 1Y Median Return: {(np.median(final)-1)*100:.1f}%')

## 5. GMM What-If Scenario

In [ ]:
shocked, normal = gmm_scenario_returns(port_ret, n_paths=500, shock_pct=-0.20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, color in [
    (axes[0], normal, 'Normal', 'blue'),
    (axes[1], shocked, '-20% Shock', 'red'),
]:
    for i in range(50):
        ax.plot(data[:, i], alpha=0.1, color=color, linewidth=0.5)
    ax.plot(np.median(data, axis=1), color='white', linewidth=2)
    ax.set_title(title); ax.set_xlabel('Days')
plt.tight_layout(); plt.show()